In [1]:
!pip install torch torchvision segmentation-models-pytorch

   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/114.6 MB ? eta -:--:--
   ---------------------------------------- 0.3/114.6 MB ? eta -:--:--
   ---------------------------------------- 1.0/114.6 MB 3.0 MB/s eta 0:00:38
    --------------------------------------- 2.1/114.6 MB 4.1 MB/s eta 0:00:28
    --------------------------------------- 2.1/114.6 MB 4.1 MB/s eta 0:00:28
    --------------------------------------- 2.1/114.6 MB 4.1 MB/s eta 0:00:28
    --------------------------------------- 2.4/114.6 MB 2.0 MB/s eta 0:00:56
    --------------------------------------- 2.4/114.6 MB 2.0 MB/s eta 0:00:56
    --------------------------------------- 2.4/114.6 MB 2.0 MB/s eta 0:00:56
    --------------------------------------- 2.6/114.6 MB 1.4 MB/s eta 0:01:20
    --------------------------------------- 2.6/114.6 MB 1.4 MB/s eta 0:01:20
   - -------------------------------------- 2.9/114.6 MB 1.3 MB/s eta 0:01:29
   - --

In [2]:
!pip install albumentations opencv-python-headless
!pip install matplotlib numpy pandas tqdm scikit-learn
!pip install kaggle pillow

   ---------------------------------------- 0.0/40.1 MB ? eta -:--:--
    --------------------------------------- 0.8/40.1 MB 4.2 MB/s eta 0:00:10
   - -------------------------------------- 1.8/40.1 MB 5.9 MB/s eta 0:00:07
   - -------------------------------------- 1.8/40.1 MB 5.9 MB/s eta 0:00:07
   -- ------------------------------------- 2.1/40.1 MB 3.1 MB/s eta 0:00:13
   -- ------------------------------------- 2.4/40.1 MB 2.4 MB/s eta 0:00:16
   -- ------------------------------------- 2.6/40.1 MB 2.0 MB/s eta 0:00:19
   -- ------------------------------------- 2.9/40.1 MB 1.8 MB/s eta 0:00:21
   -- ------------------------------------- 2.9/40.1 MB 1.8 MB/s eta 0:00:21
   --- ------------------------------------ 3.4/40.1 MB 1.7 MB/s eta 0:00:22
   --- ------------------------------------ 3.7/40.1 MB 1.7 MB/s eta 0:00:22
   ---- ----------------------------------- 4.2/40.1 MB 1.7 MB/s eta 0:00:21
   ---- ----------------------------------- 5.0/40.1 MB 1.9 MB/s eta 0:00:19
   ---

In [2]:
import os

folders = [
    'data/raw',
    'data/splits',
    'src',
    'outputs/checkpoints',
    'outputs/predictions',
    'notebooks'
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

# Create empty __init__.py for src module
with open('src/__init__.py', 'w') as f:
    pass

print("✅ All folders created!")
for f in folders:
    print(f"   📁 {f}")

✅ All folders created!
   📁 data/raw
   📁 data/splits
   📁 src
   📁 outputs/checkpoints
   📁 outputs/predictions
   📁 notebooks


In [7]:
import os
import json

# ── Paste your token from Kaggle here ──
KAGGLE_TOKEN = "KGAT_8f56a470854eb66d6d7406dec346f3e8"  # ← your token
KAGGLE_USERNAME = "sualiharajput"                  # ← your Kaggle username

# Create kaggle.json from the token
kaggle_dir = os.path.join(os.path.expanduser("~"), ".kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_json = {"sualiharajput": KAGGLE_USERNAME, "key": KAGGLE_TOKEN}

kaggle_path = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_path, "w") as f:
    json.dump(kaggle_json, f)

os.chmod(kaggle_path, 0o600)
print("✅ kaggle.json created successfully!")

✅ kaggle.json created successfully!


In [9]:
    with zipfile.ZipFile('brain-tumor-segmentation.zip', 'r') as zip_ref:
        zip_ref.extractall('data/raw/')
    print("✅ Extraction complete!")

✅ Extraction complete!


In [10]:
import os

print("📂 Contents of data/raw/:")
for root, dirs, files in os.walk('data/raw/'):
    level = root.replace('data/raw/', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}📁 {os.path.basename(root)}/ ({len(files)} files)")

# ── Auto-detect image and mask folders ──
img_dir  = None
mask_dir = None

for root, dirs, files in os.walk('data/raw/'):
    png_files = [f for f in files if f.endswith('.png') or f.endswith('.jpg')]
    if png_files:
        folder_name = os.path.basename(root).lower()
        if any(word in folder_name for word in ['image', 'img', 'scan']):
            img_dir = root
        elif any(word in folder_name for word in ['mask', 'label', 'seg', 'annotation']):
            mask_dir = root

if img_dir and mask_dir:
    print(f"\n✅ Images folder : {img_dir}")
    print(f"✅ Masks folder  : {mask_dir}")
else:
    print("\n⚠️  Could not auto-detect. Please check folder names above and set manually.")

📂 Contents of data/raw/:

⚠️  Could not auto-detect. Please check folder names above and set manually.


In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split

# ── Set these based on Cell 5 output ──
img_dir = r"C:\Users\Ismail\Documents\medical-segmentation\data\raw\images"
mask_dir = r"C:\Users\Ismail\Documents\medical-segmentation\data\raw\masks"   # adjust if different

# Get sorted file lists
all_imgs  = sorted([os.path.join(img_dir,  f) for f in os.listdir(img_dir)
                    if f.endswith('.png') or f.endswith('.jpg')])
all_masks = sorted([os.path.join(mask_dir, f) for f in os.listdir(mask_dir)
                    if f.endswith('.png') or f.endswith('.jpg')])

assert len(all_imgs) == len(all_masks), \
    f"❌ Mismatch: {len(all_imgs)} images vs {len(all_masks)} masks"

print(f"✅ Total samples: {len(all_imgs)}")

# Split: 70% train / 15% val / 15% test
train_imgs, temp_imgs, train_masks, temp_masks = train_test_split(
    all_imgs, all_masks, test_size=0.30, random_state=42)

val_imgs, test_imgs, val_masks, test_masks = train_test_split(
    temp_imgs, temp_masks, test_size=0.50, random_state=42)

# Save to CSV
pd.DataFrame({'image': train_imgs, 'mask': train_masks}).to_csv(r'C:\Users\Ismail\Documents\medical-segmentation\data\splits/train.csv', index=False)
pd.DataFrame({'image': val_imgs,   'mask': val_masks  }).to_csv(r'C:\Users\Ismail\Documents\medical-segmentation\data\splits/val.csv',   index=False)
pd.DataFrame({'image': test_imgs,  'mask': test_masks }).to_csv(r'C:\Users\Ismail\Documents\medical-segmentation\data\splits/test.csv',  index=False)

print(f"✅ Train : {len(train_imgs)} samples")
print(f"✅ Val   : {len(val_imgs)} samples")
print(f"✅ Test  : {len(test_imgs)} samples")
print("✅ Splits saved to data/splits/")

✅ Total samples: 3064
✅ Train : 2144 samples
✅ Val   : 460 samples
✅ Test  : 460 samples
✅ Splits saved to data/splits/
